In [ ]:
import re
import requests
import pandas as pd

NAVER_CLIENT_ID = "yh92px5d4h"
NAVER_CLIENT_SECRET = ""

NAVER_BLOG_API_URL = "https://naverapihub.apigw.ntruss.com/search/v1/blog"

headers = {
    "X-NCP-APIGW-API-KEY-ID": NAVER_CLIENT_ID,
    "X-NCP-APIGW-API-KEY": NAVER_CLIENT_SECRET
}

# 유틸리티 함수
def clean_text(text) :
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"&amp;", "", text)
    return text.strip()


brand_names = product_df["brand_name"].dropna().drop_duplicates().tolist()
brand_names = brand_names[:1]

# 블로그 컨텐츠 중 취합할 요소만 찾아서 컨텐츠별 저장할 딕셔너리를 모아놓기위한 용도의 리스트
all_blog_posts = list() # []

for brand_index, brand_name in enumerate(brand_names, 1) :
    params = {
        "query": brand_name,
        "display": 10,
        "start": 1,
        "sort": "date",
        "format": "json"
    }
    try :
        response = requests.get(NAVER_BLOG_API_URL, headers=headers, params=params, timeout=10)

        if response.status_code == 200 :
            data = response.json()
            items = data.get("items", [])

            for item in items :
                blog_data = {
                    "search_brand": brand_name,
                    "title": clean_text(item.get("title", "")),
                    "link": item.get("link", ""),
                    "description": clean_text(item.get("description", "")),
                    "bloggername": item.get("bloggername", ""),
                    "bloggerlink": item.get("bloggerlink", ""),
                    "postdate": item.get("postdate", "")
                }
                
                all_blog_posts.append(blog_data)
                
    except Exception as e :
        print(f"API 호출 중 오류 발생 : {e}")
        
blog_df = pd.DataFrame(all_blog_posts)  
    
# 만약 1000개 데이터를 수집하고 싶다면    
# for start in range(1, 1000, 100) :
#     params = {
#         "query": brand_name,
#         "display": 10,
#         "start": start,
#         "sort": "date",
#         "format": "json"
#     }